# Transformer Foundations, Part 3: Build the Complete Transformer Block

> **Guiding question:** Once attention can retrieve positioned context, what else must a reusable Transformer block do, and how does one prediction error train the whole path?

## 0. The Challenge: Routing Is Not the Whole Computation

Part 2 built one attention operation:

```text
positioned token vectors -> Q/K/V routing -> contextual token vectors
```

Three gaps remain:

1. One routing pattern cannot preserve several useful relationships independently.
2. Retrieving context does not decide how each token should transform what it received.
3. Deep stacks need direct information and gradient paths.

A Transformer block addresses them in one repeated shape-preserving unit:

```mermaid
flowchart LR
    X["Token states"] --> N1["Normalize"]
    N1 --> A["Multi-head attention"]
    A --> R1["Add residual"]
    X --> R1
    R1 --> N2["Normalize"]
    N2 --> F["Per-token FFN"]
    F --> R2["Add residual"]
    R1 --> R2
```

The chapter closes the complete learning path:

```text
token IDs -> embeddings + position -> Transformer blocks
-> vocabulary logits -> next-token loss -> backpropagation
```

## Fresh-Kernel Working Model

The code uses the same sentence and shape language as Parts 1–2, but rebuilds every dependency locally. Model width stays 16 through the block; four heads each work in width 4; the FFN expands to width 32 and returns to 16.

These are teaching dimensions. Production models repeat the same contracts with wider vectors, more heads, and many blocks.

In [ ]:
# Imports, tokens, embeddings, and positioned model inputs
import math
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 23
torch.manual_seed(SEED)
np.random.seed(SEED)
plt.rcParams.update({"figure.dpi": 110, "figure.facecolor": "white"})

VOCAB_PIECES = ["<PAD>", "<BOS>", "<EOS>", "the", "cat", "sat", "on", "mat"]
VOCAB = {piece: index for index, piece in enumerate(VOCAB_PIECES)}
TOKENS = ["the", "cat", "sat", "on", "the", "mat"]
TOKEN_IDS = torch.tensor([[VOCAB[token] for token in TOKENS]], dtype=torch.long)
D_MODEL, NUM_HEADS, D_HEAD, D_FF = 16, 4, 4, 32


def sinusoidal_position(sequence_length: int, width: int) -> torch.Tensor:
    positions = torch.arange(sequence_length, dtype=torch.float32).unsqueeze(1)
    pair_indices = torch.arange(0, width, 2, dtype=torch.float32)
    speeds = torch.exp(-math.log(10000.0) * pair_indices / width)
    table = torch.zeros(sequence_length, width)
    table[:, 0::2] = torch.sin(positions * speeds)
    table[:, 1::2] = torch.cos(positions * speeds)
    return table


demo_embedding = nn.Embedding(len(VOCAB), D_MODEL, padding_idx=VOCAB["<PAD>"])
x_demo = demo_embedding(TOKEN_IDS) + sinusoidal_position(len(TOKENS), D_MODEL)
print(f"token IDs {tuple(TOKEN_IDS.shape)} -> positioned vectors {tuple(x_demo.shape)}")

---

## 1. Multi-Head Attention: Keep Several Routing Views

One attention head produces one score table. If nearby syntax and a distant content relationship both matter, forcing them into one table creates a compromise.

Multi-head attention gives every head its own Q/K/V projections. Each head receives the full token state, projects to a narrower width, retrieves context, and returns its result. The head outputs are concatenated and mixed back to model width.

```text
input                         (B, S, 16)
Q/K/V split across 4 heads    (B, 4, S, 4)
score tables                  (B, 4, S, S)
concatenate + output          (B, S, 16)
```

**Predict:** Can one routing table exactly preserve both an identity pattern and a `cat`-to-`mat` relation, or must it average them? Keep your answer for the two-pattern diagnostic below.

In [ ]:
# A readable multi-head attention module
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_width = d_model // n_heads
        self.d_model = d_model
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x: torch.Tensor, causal: bool = False):
        batch, tokens, _ = x.shape
        query = self.W_Q(x).view(batch, tokens, self.n_heads, self.head_width).transpose(1, 2)
        key = self.W_K(x).view(batch, tokens, self.n_heads, self.head_width).transpose(1, 2)
        value = self.W_V(x).view(batch, tokens, self.n_heads, self.head_width).transpose(1, 2)
        scores = query @ key.transpose(-2, -1) / math.sqrt(self.head_width)
        if causal:
            future = torch.triu(torch.ones(tokens, tokens, dtype=torch.bool, device=x.device), diagonal=1)
            scores = scores.masked_fill(future, float("-inf"))
        weights = torch.softmax(scores, dim=-1)
        retrieved = weights @ value
        merged = retrieved.transpose(1, 2).contiguous().view(batch, tokens, self.d_model)
        return self.W_O(merged), weights


torch.manual_seed(SEED)
attention = MultiHeadAttention(D_MODEL, NUM_HEADS)
attention_output, head_weights = attention(x_demo)

fig, axes = plt.subplots(1, NUM_HEADS, figsize=(15, 3.5))
for head, axis in enumerate(axes):
    sns.heatmap(head_weights[0, head].detach().numpy(), ax=axis, cmap="mako", cbar=False,
                xticklabels=TOKENS, yticklabels=TOKENS, vmin=0, vmax=head_weights.max().item())
    axis.set_title(f"Head {head}"); axis.set_xlabel("Key")
    if head == 0:
        axis.set_ylabel("Query")
plt.suptitle("Random initialization already gives independent routing tables; training makes them useful")
plt.tight_layout(); plt.show()

assert attention_output.shape == x_demo.shape
print(f"head weights={tuple(head_weights.shape)}; merged output={tuple(attention_output.shape)}")
print("PASS: head split and merge preserved batch, sequence length, and model width.")

### Why more than one head can help

The next diagnostic does not claim real heads always become human-readable. It asks a narrower question: can one routing table exactly represent two conflicting target patterns at once?

- Pattern A keeps each token on itself.
- Pattern B routes `cat` to `mat` and `mat` to `cat`.

A single table must compromise. Two tables can preserve both patterns independently.

In [ ]:
# Two incompatible routing jobs, then the price of forcing both into one table
sequence_length = len(TOKENS)
identity_pattern = torch.eye(sequence_length)
relation_pattern = torch.eye(sequence_length)
cat_index, mat_index = TOKENS.index("cat"), TOKENS.index("mat")
relation_pattern[cat_index] = 0; relation_pattern[cat_index, mat_index] = 1
relation_pattern[mat_index] = 0; relation_pattern[mat_index, cat_index] = 1

one_head_compromise = (identity_pattern + relation_pattern) / 2
patterns = {"identity head": identity_pattern, "relation head": relation_pattern, "one-table compromise": one_head_compromise}
targets = {"identity target": identity_pattern, "cat<->mat target": relation_pattern}
error_table = {
    pattern_name: {target_name: F.mse_loss(pattern, target).item() for target_name, target in targets.items()}
    for pattern_name, pattern in patterns.items()
}

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for axis, matrix, title in zip(axes, patterns.values(), ["Head A: identity", "Head B: cat <-> mat", "One-head compromise"]):
    sns.heatmap(matrix.numpy(), ax=axis, cmap="Blues", vmin=0, vmax=1, cbar=False,
                xticklabels=TOKENS, yticklabels=TOKENS)
    axis.set_title(title)
plt.tight_layout(); plt.show()

print("Reconstruction error (lower means that routing job is preserved):")
for pattern_name, errors in error_table.items():
    print(f"  {pattern_name:<22} identity={errors['identity target']:.4f}  cat<->mat={errors['cat<->mat target']:.4f}")

assert error_table["identity head"]["identity target"] == 0
assert error_table["relation head"]["cat<->mat target"] == 0
assert error_table["one-table compromise"]["identity target"] > 0
assert error_table["one-table compromise"]["cat<->mat target"] > 0
print("PASS: separate heads preserve both jobs; one table pays error on both.")

---

## 2. The FFN: Process What Attention Gathered

Attention communicates **between token positions**. It does not provide a rich nonlinear computation inside each token.

The feed-forward network applies the same private processor at every position:

```text
one contextual token vector
-> expand into a wider feature workspace
-> apply a nonlinear gate
-> project back to model width
```

Memory aid: **attention communicates; the FFN computes locally.**

In [ ]:
# Shared per-token feed-forward network
class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.expand = nn.Linear(d_model, d_ff)
        self.project = nn.Linear(d_ff, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.project(F.gelu(self.expand(x)))


torch.manual_seed(SEED)
ffn = FeedForward(D_MODEL, D_FF)
expanded = ffn.expand(attention_output).detach()
ffn_output = ffn(attention_output).detach()

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
sns.heatmap(expanded[0].numpy(), ax=axes[0], cmap="RdBu_r", center=0,
            yticklabels=TOKENS, xticklabels=False)
sns.heatmap(ffn_output[0].numpy(), ax=axes[1], cmap="RdBu_r", center=0,
            yticklabels=TOKENS, xticklabels=False)
axes[0].set_title(f"Expanded private workspace: width {D_FF}")
axes[1].set_title(f"Projected back: width {D_MODEL}")
plt.tight_layout(); plt.show()

assert ffn_output.shape == attention_output.shape
print(f"attention output {tuple(attention_output.shape)} -> expanded {tuple(expanded.shape)} -> FFN output {tuple(ffn_output.shape)}")
print("PASS: the FFN changed features independently without changing sequence length or model width.")

#### The FFN works - and opens a stability problem

The FFN gave every token a private nonlinear workspace. Its outputs can also arrive with very different offsets and scales. Stack enough blocks and each sublayer must learn while its input statistics keep moving.

LayerNorm does not add context. It gives each token's next sublayer a controlled feature scale.

**Predict:** If one token vector is shifted by `+8` and another is multiplied by `5`, will LayerNorm make their feature means and spreads comparable?

In [ ]:
# One changed mechanism: raw token activations versus per-token LayerNorm
torch.manual_seed(SEED)
raw_activations = torch.randn(len(TOKENS), D_MODEL)
raw_activations[1] += 8.0
raw_activations[4] *= 5.0
layer_norm = nn.LayerNorm(D_MODEL)
normalized_activations = layer_norm(raw_activations)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for axis, matrix, title in [
    (axes[0], raw_activations, "Before LayerNorm: offsets and scales drift"),
    (axes[1], normalized_activations.detach(), "After LayerNorm: each token is conditioned"),
]:
    sns.heatmap(matrix.numpy(), ax=axis, cmap="RdBu_r", center=0, yticklabels=TOKENS,
                xticklabels=[f"d{i}" for i in range(D_MODEL)])
    axis.set_title(title)
plt.tight_layout(); plt.show()

means = normalized_activations.mean(dim=-1).detach()
stds = normalized_activations.std(dim=-1, unbiased=False).detach()
assert means.abs().max() < 1e-5 and (stds - 1).abs().max() < 1e-4
print("post-LayerNorm means:", [round(value, 4) for value in means.tolist()])
print("post-LayerNorm stds: ", [round(value, 4) for value in stds.tolist()])
print("PASS: every token enters the next sublayer with comparable feature statistics.")

---

## 3. Residual Paths and Pre-Normalization

If every block overwrote its input, useful information and gradients would be forced through every transformation. A residual path lets each sublayer propose a correction while the original stream continues directly:

```text
x -> normalize -> attention -> add to x
  -> normalize -> FFN       -> add again
```

Pre-normalization controls the input scale seen by each sublayer. Residual addition preserves a direct route through depth.

In [ ]:
# Assemble one pre-normalized Transformer block
class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attention = MultiHeadAttention(d_model, n_heads)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)

    def forward(self, x: torch.Tensor, causal: bool = False):
        attention_update, weights = self.attention(self.norm1(x), causal=causal)
        x = x + attention_update
        x = x + self.ffn(self.norm2(x))
        return x, weights


torch.manual_seed(SEED)
block = TransformerBlock(D_MODEL, NUM_HEADS, D_FF)
block_output, block_weights = block(x_demo)
input_change = (block_output - x_demo).norm(dim=-1).detach()[0]

fig, axis = plt.subplots(figsize=(8, 3.5))
axis.bar(TOKENS, input_change.numpy(), color="#2a6f97")
axis.set_title("Each token keeps the residual stream and receives a learned correction")
axis.set_ylabel("correction norm"); axis.tick_params(axis="x", rotation=25)
plt.tight_layout(); plt.show()

assert block_output.shape == x_demo.shape
print(f"input={tuple(x_demo.shape)}; block output={tuple(block_output.shape)}")
print("PASS: the complete block preserved its outer shape.")

### Residuals Preserve a Gradient Route Through Depth

This is not proof that residuals make optimization easy at any depth. It isolates one narrower claim: a skip path gives gradients a route that does not depend entirely on every learned transformation.

The next experiment reuses the same 24 learned transformations twice. The only changed variable is whether each layer adds its input back.

**Predict:** Which path leaves a larger gradient at the original input after 24 layers?

In [ ]:
# Same transformations, one variable changed: residual path on or off
torch.manual_seed(SEED)
depth = 24
transforms = nn.ModuleList([nn.Linear(D_MODEL, D_MODEL, bias=False) for _ in range(depth)])
for transform in transforms:
    nn.init.normal_(transform.weight, mean=0.0, std=0.04)


def input_gradient_norm(use_residual: bool) -> float:
    source = torch.randn(2, len(TOKENS), D_MODEL, requires_grad=True)
    state = source
    for transform in transforms:
        update = torch.tanh(transform(state))
        state = state + update if use_residual else update
    state.square().mean().backward()
    return source.grad.norm().item()


plain_gradient = input_gradient_norm(False)
residual_gradient = input_gradient_norm(True)

fig, axis = plt.subplots(figsize=(6.5, 3.7))
axis.bar(["No residual", "Residual"], [plain_gradient, residual_gradient], color=["#b91c1c", "#15803d"])
axis.set_yscale("log"); axis.set_ylabel("input gradient norm (log scale)")
axis.set_title(f"Gradient reaching the input after {depth} transformations")
plt.tight_layout(); plt.show()

assert residual_gradient > plain_gradient
print(f"no residual={plain_gradient:.3e}; residual={residual_gradient:.3e}")
print("PASS: the residual path kept a stronger gradient route through depth.")

---

## 4. From Block Outputs to Predictions

A Transformer block returns contextual vectors, not words or probabilities. A language-model head maps every final token vector to one score per vocabulary item.

During decoder training, shifted targets turn one sequence into several next-token lessons. A causal mask prevents each position from reading the answer to its own lesson.

```text
input:   <BOS> the cat sat on  the
target:  the   cat sat on  the mat
```

One scalar loss then sends gradients through the output head, FFNs, attention projections, and embeddings.

In [ ]:
# A tiny end-to-end decoder using the block we just built
class TinyTransformerLM(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, n_heads: int, d_ff: int, n_layers: int):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=VOCAB["<PAD>"])
        self.blocks = nn.ModuleList([TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_model)
        self.output_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, token_ids: torch.Tensor):
        state = self.embedding(token_ids) + sinusoidal_position(token_ids.shape[1], D_MODEL)
        attention_maps = []
        for transformer_block in self.blocks:
            state, weights = transformer_block(state, causal=True)
            attention_maps.append(weights)
        return self.output_head(self.final_norm(state)), attention_maps


torch.manual_seed(SEED)
model = TinyTransformerLM(len(VOCAB), D_MODEL, NUM_HEADS, D_FF, n_layers=2)
input_ids = torch.tensor([[VOCAB["<BOS>"], VOCAB["the"], VOCAB["cat"], VOCAB["sat"], VOCAB["on"], VOCAB["the"]]])
target_ids = torch.tensor([[VOCAB["the"], VOCAB["cat"], VOCAB["sat"], VOCAB["on"], VOCAB["the"], VOCAB["mat"]]])

logits, maps = model(input_ids)
loss = F.cross_entropy(logits.reshape(-1, len(VOCAB)), target_ids.reshape(-1))
loss.backward()

gradient_checks = {
    "embedding": model.embedding.weight.grad.norm().item(),
    "attention W_Q": model.blocks[0].attention.W_Q.weight.grad.norm().item(),
    "FFN expand": model.blocks[0].ffn.expand.weight.grad.norm().item(),
    "output head": model.output_head.weight.grad.norm().item(),
}
predicted_next = VOCAB_PIECES[logits[0, -1].argmax().item()]

assert logits.shape == (1, len(TOKENS), len(VOCAB))
assert all(value > 0 for value in gradient_checks.values())
print(f"logits={tuple(logits.shape)}; loss={loss.item():.4f}; random next-token guess={predicted_next!r}")
for component, norm in gradient_checks.items():
    print(f"  {component:<15} gradient norm={norm:.6f}")
print("PASS: one next-token loss reached embeddings, attention, the FFN, and the output head.")

---

## Chapter 3 Checkpoint

```text
token IDs
-> embeddings + position
-> [normalize -> multi-head attention -> residual
    normalize -> FFN -> residual] x depth
-> vocabulary logits
-> next-token loss
-> backpropagation through every learned stage
```

| Question | Measured answer |
|---|---|
| Why several heads? | separate tables preserved conflicting routing patterns without averaging |
| What does the FFN add? | private nonlinear feature processing at every token position |
| Why residual paths? | a stronger input gradient survived 24 transformations |
| What shape does a block preserve? | `(B, S, D)` in and `(B, S, D)` out |
| Where does prediction happen? | the output head maps contextual vectors to vocabulary logits |
| What learns from loss? | embeddings, attention, FFNs, normalization, and the head share one graph |

**Your turn:** Change `NUM_HEADS` from 4 to 2 while keeping `D_MODEL=16`. Predict the new head width and score-table shape before rerunning the block.

The reusable block is ready. [Part 4 — Decoder-Only Language Models](04-decoder-only-language-model.ipynb) adds the causal training/generation contract, shifted targets, sampling, and KV-cache behavior.